In [1]:
%run nb_dq_utils

StatementMeta(, 7075123e-65c0-4a9f-ae35-397545782e56, 10, Finished, Available, Finished, True)

In [2]:
from pyspark.sql import functions as F
 
logger = get_logger("silver.address.dq") 

# ── Read Bronze ───────────────────────────────────────────────────────
df_address = spark.read.table("lh_adventureworks_bronze.dbo.Person_Address")
expected_rows = df_address.count()   
logger.info(f"Address rows:{expected_rows}")
df_address.printSchema()

try:
    df_stateprovince = spark.read.table("lh_adventureworks_bronze.dbo.Person_StateProvince")
    has_stateprovince = True
    logger.info(f"StateProvince rows: {df_stateprovince.count()}")
    logger.info(f"StateProvince rows: {df_stateprovince.count()}")
except Exception:
    has_stateprovince = False
    logger.info("Note: Person_StateProvince not in Bronze")

StatementMeta(, 7075123e-65c0-4a9f-ae35-397545782e56, 11, Finished, Available, Finished, False)

root
 |-- AddressID: integer (nullable = true)
 |-- AddressLine1: string (nullable = true)
 |-- AddressLine2: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StateProvinceID: integer (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- SpatialLocation: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



INFO:silver.address.dq:StateProvince rows: 181
INFO:silver.address.dq:StateProvince rows: 181


In [3]:
from pyspark.sql import functions as F

# ── Join Address + StateProvince ──────────────────────────────────────
df_sp_clean = df_stateprovince.select(
    F.col("StateProvinceID").alias("sp_StateProvinceID"),
    F.col("StateProvinceCode"),
    F.col("Name").alias("StateProvinceName"),
    F.col("CountryRegionCode"),
    F.col("TerritoryID").alias("sp_TerritoryID")
)

df_silver_address = df_address.join(
    df_sp_clean,
    df_address["StateProvinceID"] == df_sp_clean["sp_StateProvinceID"],
    how="left"
).drop("sp_StateProvinceID")

# ── Select final columns ──────────────────────────────────────────────
df_silver_address = df_silver_address.select(
    "AddressID",
    "AddressLine1",
    "AddressLine2",
    "City",
    "StateProvinceID",
    "StateProvinceCode",
    "StateProvinceName",
    "CountryRegionCode",
    F.col("sp_TerritoryID").alias("TerritoryID"),
    "PostalCode",
    "SpatialLocation",
    "rowguid",
    "ModifiedDate"
)

actual_rows = df_silver_address.count()
logger.info(f"Row count: {actual_rows}")
logger.info(f"{df_silver_address.printSchema()}")

StatementMeta(, 7075123e-65c0-4a9f-ae35-397545782e56, 12, Finished, Available, Finished, False)

root
 |-- AddressID: integer (nullable = true)
 |-- AddressLine1: string (nullable = true)
 |-- AddressLine2: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StateProvinceID: integer (nullable = true)
 |-- StateProvinceCode: string (nullable = true)
 |-- StateProvinceName: string (nullable = true)
 |-- CountryRegionCode: string (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- SpatialLocation: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



In [4]:
# ── DQ check ──────────────────────────────────────────────────────────────
checks = [
    {
        "name": "Row count",
        "passed": actual_rows == expected_rows,
        "message": f"Expected {expected_rows:,}, got {actual_rows:,}"
    },
    check_null_pk(df_silver_address, "AddressID"),
    check_duplicate_pk(df_silver_address, "AddressID"),
    check_join_coverage(df_silver_address, "StateProvinceID", "StateProvinceName",
                         label="StateProvince join coverage"),
]
 
run_dq_checks(checks, logger, "silver.address")

StatementMeta(, 7075123e-65c0-4a9f-ae35-397545782e56, 13, Finished, Available, Finished, False)

INFO:silver.address.dq:[DQ] [PASS] Row count — Expected 19,614, got 19,614
INFO:silver.address.dq:[DQ] [PASS] Null PK — AddressID — 0 nulls
INFO:silver.address.dq:[DQ] [PASS] Duplicate PK — AddressID — 0 duplicates
INFO:silver.address.dq:[DQ] [PASS] StateProvince join coverage — 0 rows with StateProvinceID but no StateProvinceName
INFO:silver.address.dq:[DQ] All checks passed for silver.address.


In [5]:
# ── Write ──────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.person")

df_silver_address.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.person.address")

df_verify = spark.read.table("lh_adventureworks_silver.person.address")
logger.info(f"silver.person.address written — {df_verify.count():,} rows verified.")
df_silver_address.unpersist()

StatementMeta(, 7075123e-65c0-4a9f-ae35-397545782e56, 14, Finished, Available, Finished, False)

INFO:silver.address.dq:silver.person.address written — 19,614 rows verified.


DataFrame[AddressID: int, AddressLine1: string, AddressLine2: string, City: string, StateProvinceID: int, StateProvinceCode: string, StateProvinceName: string, CountryRegionCode: string, TerritoryID: int, PostalCode: string, SpatialLocation: string, rowguid: string, ModifiedDate: timestamp]